# Enzyme Function Prediction Using Graph Neural Networks and Protein Structure
This notebook is a toy project for learning Graph Neural Networks and applying them to protein structure. More precisely, given the structure of an enzyme this model will predict if it is an oxyreductase or a transferase.

##1. Data Manipulation

The protein structures were downloaded from RCSB PDB in format mmCIF. We selected proteins from Escherichia coli with structure experimentally determined by X-ray diffraction with a refinement resolution of at most 2.5Å having Enzyme Comission number exclusively 1 or exclusively 2.

###1.1 Obtaining $\alpha$-carbons scaffold of the structure
The $C_{\alpha}$ of an amino acid can be viewed as its "central" point, keeping only these carbons in the structure of the protein can be seen as a simplification that retains the essential information about the molecule structure.

In [ ]:
from Bio.PDB import MMCIFParser

#This function return the list of the spatial coordinates of the alpha-carbons
#of a given mmCIF file. It uses a mmCIF parser from Biopython.
def get_alpha_carbons(cif_file):
  parser = MMCIFParser(QUIET=True)
  structure = parser.get_structure("protein", cif_file)

  ca_coords = []

  for model in structure:
    for chain in model:
      for residue in chain:
        if "CA" in residue:
          atom = residue["CA"]
          coord = atom.get_coord()
          aa = residue.resname
          ca_coords.append((aa,coord))

  return ca_coords

In [ ]:
import os

#(Truncated) output from the above defined function
get_alpha_carbons('data/EC_01/1aa6.cif')[:5]

[('MET', array([68.592,  3.44 , 30.52 ], dtype=float32)),
 ('LYS', array([71.858,  4.86 , 31.889], dtype=float32)),
 ('LYS', array([73.133,  8.159, 33.285], dtype=float32)),
 ('VAL', array([76.264,  9.713, 31.806], dtype=float32)),
 ('VAL', array([77.941, 12.384, 33.981], dtype=float32))]

###1.2 Creating graph representation

We now create a graph that represents the protein structure, the nodes will be $C_{\alpha}$ and there will be an edge betwen two carbons when their distance is less than a defined distance cutoff. The graph will be represented using edge index.

In [ ]:
import numpy as np

def generate_graph(ca_list, dist_cutoff=8):

  edge_index_start = []
  edge_index_end = []

  N = len(ca_list)

  for i in range(N):
    for j in range(i,N):
      if np.linalg.norm(ca_list[i][1]-ca_list[j][1]) < dist_cutoff:
        edge_index_start.append(i)
        edge_index_end.append(j)

        if i != j:
          edge_index_start.append(j)
          edge_index_end.append(i)

  edge_index = np.array([edge_index_start, edge_index_end])
  return edge_index

In [ ]:
#(Truncated) example of an edge list
ca_list = get_alpha_carbons('data/EC_01/1aa6.cif')
a, b = generate_graph(ca_list)

for i,j in zip(a,b):
  print(i,j)

  if i > 5:
    break

0 0
0 1
1 0
0 2
2 0
0 19
19 0


In [ ]:
from tqdm import tqdm

data_dirs = ['data/EC_01', 'data/EC_02']
all_alpha_carbons_01 = []
all_alpha_carbons_02 = []

for data_dir in data_dirs:
  print(f"Processing files in {data_dir}")
  for filename in tqdm(os.listdir(data_dir)):
    file_path = os.path.join(data_dir, filename)
    ca_coords = get_alpha_carbons(file_path)
    pdb_id = file_path[-8:-4]
    if data_dir == 'data/EC_01':
      all_alpha_carbons_01.append((pdb_id, ca_coords))
    elif data_dir == 'data/EC_02':
      all_alpha_carbons_02.append((pdb_id, ca_coords))

Processing files in data/EC_01


100%|██████████| 471/471 [03:45<00:00,  2.09it/s]


Processing files in data/EC_02


100%|██████████| 752/752 [05:18<00:00,  2.36it/s]


In [ ]:
import pickle

output_dir = 'data_processed'
os.makedirs(output_dir, exist_ok=True)

output_path_01 = os.path.join(output_dir, 'all_alpha_carbons_01.pkl')
with open(output_path_01, 'wb') as f:
    pickle.dump(all_alpha_carbons_01, f)
print(f"Saved all_alpha_carbons_01 to {output_path_01}")

output_path_02 = os.path.join(output_dir, 'all_alpha_carbons_02.pkl')
with open(output_path_02, 'wb') as f:
    pickle.dump(all_alpha_carbons_02, f)
print(f"Saved all_alpha_carbons_02 to {output_path_02}")

Saved all_alpha_carbons_01 to data_processed/all_alpha_carbons_01.pkl
Saved all_alpha_carbons_02 to data_processed/all_alpha_carbons_02.pkl


In [ ]:
graphs_01 = []
print("Generating graphs for oxireductases.")
for pdb_id, ca_coords in tqdm(all_alpha_carbons_01):
  if ca_coords:
    edge_index = generate_graph(ca_coords)
    graphs_01.append((pdb_id, edge_index))
  else:
    print(f"Warning: No alpha carbons found for {pdb_id} in EC_01. Skipping graph generation.")

Generating graphs for oxireductases.


100%|██████████| 471/471 [19:43<00:00,  2.51s/it]


In [ ]:
graphs_02 = []
print("Generating graphs for transferases.")
for pdb_id, ca_coords in tqdm(all_alpha_carbons_02):
  if ca_coords:
    edge_index = generate_graph(ca_coords)
    graphs_02.append((pdb_id, edge_index))
  else:
    print(f"Warning: No alpha carbons found for {pdb_id} in EC_02. Skipping graph generation.")

Generating graphs for transferases.


100%|██████████| 752/752 [19:03<00:00,  1.52s/it]


In [ ]:
output_path_graphs_01 = os.path.join(output_dir, 'graphs_01.pkl')
with open(output_path_graphs_01, 'wb') as f:
  pickle.dump(graphs_01, f)
print(f"Saved graphs_01 to {output_path_graphs_01}")

output_path_graphs_02 = os.path.join(output_dir, 'graphs_02.pkl')
with open(output_path_graphs_02, 'wb') as f:
  pickle.dump(graphs_02, f)
print(f"Saved graphs_02 to {output_path_graphs_02}")

Saved graphs_01 to data_processed/graphs_01.pkl
Saved graphs_02 to data_processed/graphs_02.pkl


## 2. Data Manipulation - Making the data ready for Machine Learning

We need everything ready for Pytorch.

Given a graph $G$ with $n$ vertices, each vertex is represented by a number from $0$ to $n-1$.

Each vertex is associated with a feature vector that contains information about this vertex, let $m$ be the dimension of these vectors. The complete set of features will be represented by a Pytorch tensor $H$ of dimensions $n \times m$, where each line $H_{i,:}$ is the vector of size $m$ associated with vertex $i$.

The edges of the graph will be represented by an edge list, a tensor of size $e \times 2$ where $e$ is the number of edges in the graph, when a column has entries $i$ and $j$ it means that these vertices are connected in the graph.



### 2.1 Encondig vertex features

Proteins are composed of amino acids linked by peptide bonds, each vertex of our graph corresponds from an alpha-carbon from an amino acid. The initial feature vector of each vertex will encode the type of the corresponding amino acid. We will use one-hot encoding in 21 classes: one for each canonical amino acid and one for non-canonical amino-acids.

The function below takes as input a list of $n$ amino acids of a protein, and outputs a $n \times 21$ matrix that is the matrix of the features of the vertices of the associated graph.

In [57]:
import torch
import torch.nn.functional as F

def one_hot_encoding(aa_list):
  canonical_aa = [
          'GLY', 'ALA', 'VAL', 'LEU', 'ILE', 'PRO', 'MET',
          'PHE', 'TRP', 'TYR', 'SER', 'THR', 'CYS', 'ASN',
          'GLN', 'LYS', 'ARG', 'HIS', 'ASP', 'GLU'
      ]

  aa_to_idx = {aa:i for i, aa in enumerate(canonical_aa)}

  indices = torch.tensor([aa_to_idx.get(aa.upper(), 20) for aa in aa_list], dtype=torch.long)

  one_hot = F.one_hot(indices, num_classes=21)

  return one_hot

Since we have 471 hydrolase examples and 752 transferase examples, we will undersample to obtain balanced classes. We will select 350, 50 and 50 examples from each class for the training, validation and testing datasets.

In [87]:
import random

n_train, n_val, n_test = 350, 50, 50

def get_split_idx(total, n_train, n_val, n_test):
  indices = list(range(total))
  random.shuffle(indices)

  train_idx = indices[:n_train]
  val_idx = indices[n_train:n_train+n_val]
  test_idx = indices[n_train+n_val:n_train+n_val+n_test]

  return train_idx, val_idx, test_idx

Now we will create the datasets. PyTorch Geometric has the class Data for storing graphs, each dataset will be a list of Data objects.

In [106]:
from torch_geometric.data import Data

hydrolase_idx = get_split_idx(len(all_alpha_carbons_01), n_train, n_val, n_test)
transferase_idx = get_split_idx(len(all_alpha_carbons_02), n_train, n_val, n_test)

testing_dataset = []
validation_dataset = []
training_dataset = []

for i in tqdm(range(len(all_alpha_carbons_01))):

  assert all_alpha_carbons_01[i][0] == graphs_01[i][0]
  graph = Data(
      x = one_hot_encoding([all_alpha_carbons_01[i][1][_][0] for _ in range(len(all_alpha_carbons_01[i][1]))]),
      edge_index = torch.from_numpy(graphs_01[i][1]),
      y=0
  )

  if i in hydrolase_idx[0]:
    training_dataset.append(graph)
  elif i in hydrolase_idx[1]:
    validation_dataset.append(graph)
  elif i in hydrolase_idx[2]:
    testing_dataset.append(graph)

for i in tqdm(range(len(all_alpha_carbons_02))):

  assert all_alpha_carbons_02[i][0] == graphs_02[i][0]
  graph = Data(
      x = one_hot_encoding([all_alpha_carbons_02[i][1][_][0] for _ in range(len(all_alpha_carbons_02[i][1]))]),
      edge_index = torch.from_numpy(graphs_02[i][1]),
      y=1
  )

  if i in transferase_idx[0]:
    training_dataset.append(graph)
  elif i in transferase_idx[1]:
    validation_dataset.append(graph)
  elif i in transferase_idx[2]:
    testing_dataset.append(graph)

random.shuffle(training_dataset)
random.shuffle(validation_dataset)
random.shuffle(testing_dataset)

100%|██████████| 752/752 [00:00<00:00, 1449.49it/s]


Having the datasets, we can create the data loaders.

In [107]:
from torch_geometric.loader import DataLoader

batch_size = 32

training_loader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
validation_loader   = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
testing_loader  = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)